# Phase 1 — Training: Baseline (A1) vs Full Model (A4)

**Goal:** Train and compare two models on Towards Comprehensive data.

| Model | Features | Purpose |
|-------|----------|---------|
| **A1 — Baseline** | Text only (TF-IDF + XGBoost) | How good is message text alone? |
| **A4 — Full model** | Text + Peerness + Repetition | Does adding relational/behavioural context help? |

**Code:** `src/phase1_training/models/train.py`

## 1. Training setup

| Setting | Value | Why |
|---------|-------|-----|
| **Split** | 80% train / 20% test | Standard hold-out evaluation |
| **Split unit** | `(User1, User2)` pairs | Same pair never in both train and test (no leakage) |
| **Stratification** | By pair-level max `CBLabel` | Preserve rare cyberbullying cases in both splits |
| **Text pipeline** | Lowercase → remove URLs/mentions → TF-IDF | Standard NLP preprocessing |
| **Class imbalance** | `scale_pos_weight ≈ 128` | Up-weight minority class in XGBoost |
| **Tabular features (A4)** | Peerness, TotalMsgs, AggressiveMsgs, aggression_ratio | Relational + behavioural signals |

### Models

```
A1 (baseline):  Message → TF-IDF → XGBoost → prediction
A4 (full):      Message → TF-IDF ──┐
                                   ├→ concat → XGBoost → prediction
                Peerness, repetition ┘
```

## 2. Run training

This cell trains both models and saves outputs. Training takes ~5–15 seconds.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "src").is_dir() and (candidate / "requirements.txt").is_file():
        ROOT = candidate
        break
else:
    raise RuntimeError(
        "Cannot find F21MP project root. Open the F21MP folder in Cursor/VS Code."
    )

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.phase1_training.models.train import train_phase1

results = train_phase1(test_size=0.2, random_state=42, save_outputs=True, verbose=True)

Total samples: 15501
Train: 13031 | Test: 2470
Train pairs: 6691
Test pairs:  1673
Train CBLabel counts:
CBLabel
0    12930
1      101
Name: count, dtype: int64
scale_pos_weight: 128.02

=== Baseline A1 (text-only) ===
Accuracy:  0.9660
Precision: 0.0000
Recall:    0.0000
F1:        0.0000
ROC-AUC:   0.5095
Confusion matrix: [[2386, 59], [25, 0]]

=== Full model A4 (text + peerness + repetition) ===
Accuracy:  0.9996
Precision: 0.9615
Recall:    1.0000
F1:        0.9804
ROC-AUC:   1.0000
Confusion matrix: [[2444, 1], [0, 25]]

Models saved to: /Users/pharshitha/Desktop/F21MP/outputs/models
Metrics saved to: /Users/pharshitha/Desktop/F21MP/outputs/results/phase1_test_metrics.json


## 3. Phase 1 results (test set)

Expected results on the **20% held-out test split** (2,470 messages, 25 cyberbullying):

| Model | F1 | Precision | Recall | ROC-AUC | Verdict |
|-------|-----|-----------|--------|---------|---------|
| **A1 — Text only** | 0.00 | 0.00 | 0.00 | 0.51 | Fails to detect cyberbullying — predicts almost all as non-CB |
| **A4 — Full model** | **0.98** | **0.96** | **1.00** | **1.00** | Strong detection using text + peerness + repetition |

### Confusion matrices

**A1 (baseline)** — `[[TN, FP], [FN, TP]]` = `[[2386, 59], [25, 0]]`
- **25 false negatives** — all cyberbullying cases missed
- **59 false positives** — non-cyberbullying flagged incorrectly
- ROC-AUC ≈ 0.51 → no better than random for ranking positives

**A4 (full model)** — `[[2444, 1], [0, 25]]`
- **25/25 cyberbullying cases caught** (recall = 100%)
- Only **1 false positive** out of 2,445 non-cyberbullying messages

### What this means for the research question

> *Do peerness and repetition help compared with text-only?*

**Early answer: yes.** Text alone (A1) cannot handle the severe class imbalance on this dataset. Adding peerness and repetition (A4) gives a large improvement. Formal ablation (A2, A3) in Phase 3 will quantify each feature's contribution separately.

### Saved outputs

| File | Location |
|------|----------|
| Baseline model | `outputs/models/baseline_a1.joblib` |
| Full model | `outputs/models/full_a4.joblib` |
| Metrics JSON | `outputs/results/phase1_test_metrics.json` |
| Confusion matrix plots | `outputs/media/confusion_matrix_a1.png`, `confusion_matrix_a4.png` |

## 4. Reload metrics from file

Confirm the saved JSON matches the printed output above.

In [2]:
import json
from pathlib import Path

metrics_path = ROOT / "outputs" / "results" / "phase1_test_metrics.json"
with open(metrics_path) as f:
    metrics = json.load(f)

for name, m in metrics.items():
    print(f"\n{name}:")
    print(f"  F1={m['f1']:.4f}  Precision={m['precision']:.4f}  Recall={m['recall']:.4f}  ROC-AUC={m.get('roc_auc')}")


baseline_a1:
  F1=0.0000  Precision=0.0000  Recall=0.0000  ROC-AUC=0.5094969325153375

full_a4:
  F1=0.9804  Precision=0.9615  Recall=1.0000  ROC-AUC=0.9999836400817995


## 5. Phase 1 training — complete ✓

| Task | Status |
|------|--------|
| Load & merge data | ✓ (notebook 01) |
| Feature engineering | ✓ |
| Text preprocessing + TF-IDF | ✓ |
| Baseline A1 (text-only) | ✓ |
| Full model A4 (text + peerness + repetition) | ✓ |
| Pair-aware stratified split | ✓ |
| Evaluation metrics + plots | ✓ |

**What's next (Phase 2):** Validate on independent Twitter/YouTube data — implement `src/phase2_validation/evaluation/validation.py`